In [21]:
import pandas as pd
import requests

# Exploration

In [22]:
url = "https://api.jolpi.ca/ergast/f1/current/drivers.json"
response = requests.get(url)

if response.status_code == 200:
    data = response.json()
    liste_pilotes = data["MRData"]["DriverTable"]["Drivers"]
    df_drivers = pd.DataFrame(liste_pilotes)
    display(df_drivers)
else:
    print(f"Erreur : {response.status_code}")

,driverId,permanentNumber,code,url,givenName,familyName,dateOfBirth,nationality
0,albon,23,ALB,http://en.wikipedia.org/wiki/Alexander_Albon,Alexander,Albon,1996-03-23,Thai
1,alonso,14,ALO,http://en.wikipedia.org/wiki/Fernando_Alonso,Fernando,Alonso,1981-07-29,Spanish
2,antonelli,12,ANT,https://en.wikipedia.org/wiki/Andrea_Kimi_Anto...,Andrea Kimi,Antonelli,2006-08-25,Italian
3,paul_aron,NaN,NaN,NaN,Paul,Aron,NaN,NaN
4,bearman,87,BEA,http://en.wikipedia.org/wiki/Oliver_Bearman,Oliver,Bearman,2005-05-08,British
5,dino_beganovic,NaN,NaN,NaN,Dino,Beganovic,NaN,NaN
6,bortoleto,5,BOR,https://en.wikipedia.org/wiki/Gabriel_Bortoleto,Gabriel,Bortoleto,2004-10-14,Brazilian
7,bottas,77,BOT,http://en.wikipedia.org/wiki/Valtteri_Bottas,Valtteri,Bottas,1989-08-28,Finnish
8,luke_browning,NaN,NaN,NaN,Luke,Browning,NaN,NaN
9,colapinto,43,COL,http://en.wikipedia.org/wiki/Franco_Colapinto,Franco,Colapinto,2003-05-27,Argentine


# Transformation

In [23]:
df_drivers.to_parquet('drivers_latest.parquet', index = False)
print("Fichier parquet généré.")

Fichier parquet généré.


# Chargement

In [24]:
from azure.storage.blob import BlobServiceClient
from dotenv import load_dotenv
import os

In [25]:
load_dotenv()

connection_string = os.getenv("AZURE_CONNECTION_STRING")
if not connection_string:
    raise ValueError("Erreur de lecture du fichier .env")

container_name = "bronze"
local_file_name = "drivers_latest.parquet"

blob_name = "drivers/drivers_latest.parquet"

In [26]:
blob_service_client = BlobServiceClient.from_connection_string(connection_string)
blob_client = blob_service_client.get_blob_client(container=container_name, blob=blob_name)
print("Debut de l'envoi vers le data lake")

with open(local_file_name, "rb") as data:
    blob_client.upload_blob(data, overwrite=True)

print("Fichier Parquet envoyé avec succès dans le dossier bronze")

Debut de l'envoi vers le data lake
Fichier Parquet envoyé avec succès dans le dossier bronze


In [27]:
print(df_drivers.columns)

Index(['driverId', 'permanentNumber', 'code', 'url', 'givenName', 'familyName',
       'dateOfBirth', 'nationality'],
      dtype='str')
